In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import scvi
import scanpy as sc
from datasets.cross_tissue_atlas import CrossTissueDataset

# silence all warnings
import warnings
warnings.filterwarnings("ignore")

/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [3]:
test_set = CrossTissueDataset(
    root="data",
    split="test",
)

In [4]:
adata = test_set.adata

In [5]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=500,
    flavor="seurat",
    batch_key="donor_id",
    subset=True,
)

scvi.model.SCVI.setup_anndata(
    adata,
    batch_key="donor_id",
)

model = scvi.model.SCVI(
    adata,
    n_latent=10,
    n_layers=2,
)

model.train(max_epochs=100, batch_size=512)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


In [13]:
# save model
model.save("outputs/batch_scvi_model", overwrite=True)

In [6]:
from geomloss import SamplesLoss
import torch
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [7]:
# fit nearest neighbors on adata
nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(adata.X)

energy = SamplesLoss("energy")

In [14]:
# Precompute all transformations for each donor pair
from collections import defaultdict

energy_dists = []

for p in range(8):
    # Collect all source indices and target info for this partition
    batches_info = []
    for _ in range(100):
        batch = test_set[p]
        batches_info.append(batch)
    
    # Group by target donor to batch the scVI calls
    donor_groups = defaultdict(list)
    for i, batch in enumerate(batches_info):
        target_donor = batch['target_metadata']['donor_id']
        donor_groups[target_donor].append((i, batch))
    
    # Process each donor group together
    results = [None] * 100
    for target_donor, group in donor_groups.items():
        # Combine all source indices for this target donor
        all_source_inds = np.concatenate([b['source_metadata']['adata_indices'] for _, b in group])
        adata_source = adata[all_source_inds].copy()
        
        # Single scVI call for all samples going to this donor
        transformed = model.get_normalized_expression(
            adata_source, batch_size=512, transform_batch=target_donor
        )
        
        # Split results back
        offset = 0
        for i, batch in group:
            n_samples = len(batch['source_metadata']['adata_indices'])
            results[i] = (transformed[offset:offset+n_samples], batch['target_samples'])
            offset += n_samples
    
    # Compute energy distances
    for transformed, target_samples in results:
        nns = nbrs.kneighbors(transformed, return_distance=False)
        snapped = adata.obsm['X_pca'][nns.flatten()]
        e_dist = energy(torch.tensor(snapped), target_samples.squeeze(0)).item()
        energy_dists.append(e_dist)

INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer set

In [18]:
print("mean Energy Distance:", np.mean(energy_dists))
print('s.e.m', np.std(energy_dists) / np.sqrt(len(energy_dists)))

mean Energy Distance: 4.822914915680886
s.e.m 0.016568336394916908
